# SymPy: символьные вычисления в научных задачах

**Проект «ИИ для учёных».** Практический блокнот к разделу «Инструменты».

## Что делает этот инструмент

SymPy — библиотека символьной математики на Python: работает с формулами аналитически, не численно. Она может упрощать выражения, брать производные и интегралы в точном виде, решать уравнения и системы, вычислять пределы, раскладывать функции в ряды Тейлора и выполнять матричные операции.

Для исследователя это значит: рутинные аналитические выкладки, которые раньше требовали карандаша и бумаги (или дорогого CAS-пакета), теперь доступны прямо в Python-скрипте. SymPy особенно ценен там, где нужна точная формула, а не число: при выводе уравнений движения, проверке аналитических решений, генерации кода из математических выражений.

В этом блокноте мы разберём ключевые возможности на примерах из физики и математики, и закончим полным аналитическим выводом: символьное решение → числовая функция → график. Расчётное время прохождения — около пяти минут.

## 1. Установка библиотек

In [ ]:
!pip install -q sympy matplotlib numpy

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()

# Красивый вывод LaTeX в Jupyter/Colab
from sympy import init_printing
init_printing(use_latex="mathjax")
print("Стиль графиков и LaTeX-вывод подключены.")

## 3. Символы, выражения и упрощение

Первый шаг в SymPy — объявить символы. Только после этого Python начинает обращаться с переменными как с математическими символами, а не числами.

In [ ]:
from sympy import (
    symbols, sqrt, exp, sin, cos, pi, I, oo,
    simplify, expand, factor, cancel,
    solve, dsolve, Eq,
    diff, integrate, limit, series,
    Matrix, eye,
    lambdify, latex,
    Rational
)

# Объявляем символы; assumptions (positive=True) помогают simplify
x, y, z = symbols("x y z")
t       = symbols("t", real=True, positive=True)
omega   = symbols("omega", real=True, positive=True)
m, k, A = symbols("m k A", positive=True)

# Основное выражение: (x + 1)^2 - 1
expr = (x + 1)**2 - 1
print("Исходное выражение:")
display(expr)

print("\nРаскрыть скобки (expand):")
display(expand(expr))

print("\nРазложить на множители (factor):")
display(factor(expand(expr)))

In [ ]:
# simplify: SymPy сам выбирает подходящее упрощение
expr2 = (x**2 - 1) / (x - 1)
print("Выражение до упрощения:")
display(expr2)
print("После simplify:")
display(simplify(expr2))

# cancel: сокращение дроби
print("\ncancel — сокращение алгебраической дроби:")
display(cancel((x**2 + 2*x + 1) / (x + 1)))

## 4. Решение уравнений и систем

`solve` находит аналитическое решение алгебраических уравнений. Для дифференциальных уравнений используется `dsolve`.

In [ ]:
# Квадратное уравнение ax^2 + bx + c = 0
a, b, c = symbols("a b c")
quad = a*x**2 + b*x + c
roots = solve(quad, x)
print("Корни квадратного уравнения ax² + bx + c = 0:")
for r in roots:
    display(r)

# Система линейных уравнений
print("\nСистема: x + y = 5, x - y = 1")
sol = solve([x + y - 5, x - y - 1], [x, y])
display(sol)

## 5. Производные и интегралы

`diff` берёт производную любого порядка. `integrate` — неопределённый или определённый интеграл. Всё в точном аналитическом виде.

In [ ]:
f = sin(x) * exp(-x)

print("f(x) = sin(x) * exp(-x)")
print("\nПервая производная f'(x):")
df = diff(f, x)
display(df)

print("\nВторая производная f''(x):")
display(diff(f, x, 2))

print("\nНеопределённый интеграл ∫f dx:")
display(integrate(f, x))

print("\nОпределённый интеграл ∫₀^∞ f dx:")
display(integrate(f, (x, 0, oo)))

## 6. Пределы и ряды Тейлора

Пределы вычисляются точно, в том числе неопределённые формы. Ряды Тейлора удобны для локального анализа и численных приближений.

In [ ]:
# Неопределённые формы 0/0 и ∞ → SymPy вычисляет правильно
print("Предел sin(x)/x при x → 0 (форма 0/0):")
display(limit(sin(x)/x, x, 0))

print("\nПредел (1 + 1/x)^x при x → ∞ (число e):")
display(limit((1 + 1/x)**x, x, oo))

# Ряд Тейлора: sin(x) в окрестности 0 до 7-го порядка
print("\nРяд Тейлора sin(x) вокруг x=0, до O(x^7):")
taylor_sin = series(sin(x), x, 0, 7)
display(taylor_sin)

## 7. Работа с матрицами

SymPy выполняет матричные операции аналитически: нахождение собственных значений, обращение, детерминант — всё в точном виде.

In [ ]:
lam = symbols("lambda")

# Матрица 2×2 с символьными элементами
M = Matrix([
    [1,  2],
    [3, -1],
])

print("Матрица M:")
display(M)

print("\nДетерминант:")
display(M.det())

print("\nОбратная матрица M⁻¹:")
display(M.inv())

print("\nСобственные значения (eigenvals → {значение: кратность}):")
display(M.eigenvals())

print("\nСобственные векторы (eigenvects → [(λ, кратность, [вектор])]):")
evects = M.eigenvects()
for lam_val, mult, vecs in evects:
    print(f"  λ = ", end=""); display(lam_val)
    print(f"  вектор: ", end=""); display(vecs[0])

## 8. Физический пример: гармонический осциллятор

Выведем аналитически закон движения груза на пружине (гармонический осциллятор без затухания). Уравнение движения:

$$m \ddot{x}(t) + k\, x(t) = 0$$

где $m$ — масса, $k$ — жёсткость пружины, $x(t)$ — отклонение от положения равновесия.

SymPy решит это ОДУ аналитически, применит начальные условия $x(0) = A$, $\dot{x}(0) = 0$, а затем `lambdify` превратит символьное решение в быструю числовую функцию для построения графика.

In [ ]:
from sympy import Function, dsolve, Eq, diff as Diff, symbols, cos, sqrt
from sympy import simplify

# Символьная функция x(t) — неизвестное перемещение
x_fn = Function("x")
t_sym = symbols("t", positive=True)
m_s, k_s, A_s = symbols("m k A", positive=True)

# Уравнение движения: m*x''(t) + k*x(t) = 0
ode = Eq(m_s * Diff(x_fn(t_sym), t_sym, 2) + k_s * x_fn(t_sym), 0)
print("Уравнение движения:")
display(ode)

# Общее решение (C1, C2 — произвольные константы)
general_sol = dsolve(ode, x_fn(t_sym))
print("\nОбщее решение:")
display(general_sol)

In [ ]:
from sympy import symbols as syms, cos, sqrt, solve as sym_solve, lambdify

# Применяем начальные условия: x(0) = A, x'(0) = 0
# Общее решение: x(t) = C1*cos(omega*t) + C2*sin(omega*t), omega = sqrt(k/m)
C1, C2 = syms("C1 C2")
omega_s = sqrt(k_s / m_s)

# Явно строим x(t) с произвольными константами
x_general = C1 * cos(omega_s * t_sym) + C2 * (1/omega_s) * (-omega_s * cos(omega_s * t_sym)).diff(t_sym).subs(t_sym, 0)
# Корректнее: x = C1*cos(w*t) + C2*sin(w*t)
from sympy import sin
x_gen = C1 * cos(omega_s * t_sym) + C2 * sin(omega_s * t_sym)
x_prime = Diff(x_gen, t_sym)

# Начальные условия
ic_eqs = [
    x_gen.subs(t_sym, 0) - A_s,        # x(0) = A
    x_prime.subs(t_sym, 0),              # x'(0) = 0
]
consts = sym_solve(ic_eqs, [C1, C2])
x_particular = x_gen.subs(consts)
x_particular = simplify(x_particular)

print("Частное решение при x(0)=A, x'(0)=0:")
display(x_particular)
print("\nВ формате LaTeX:")
print(latex(x_particular))

### Шаг 8.3. lambdify: символьная формула → числовая функция → график

`lambdify` компилирует символьное выражение SymPy в быструю numpy-функцию. Это стандартный мост между аналитическим выводом и численным использованием.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Параметры конкретной системы
A_val     = 1.0    # амплитуда [м]
m_val     = 0.5    # масса [кг]
k_val     = 2.0    # жёсткость [Н/м]
omega_val = np.sqrt(k_val / m_val)  # собственная частота [рад/с]

# lambdify: x_particular(t_sym; m_s, k_s, A_s) → функция Python
x_num = lambdify(
    [t_sym, m_s, k_s, A_s],
    x_particular,
    modules="numpy",
)

t_arr = np.linspace(0, 4 * np.pi / omega_val, 500)
x_arr = x_num(t_arr, m_val, k_val, A_val)

# Скорость: производная аналитически, затем тоже lambdify
v_sym = Diff(x_particular, t_sym)
v_num = lambdify([t_sym, m_s, k_s, A_s], v_sym, modules="numpy")
v_arr = v_num(t_arr, m_val, k_val, A_val)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

# График x(t)
axes[0].plot(t_arr, x_arr, color=VIZ["series"][0],
             label=f"x(t) = A·cos(ωt),  ω = {omega_val:.2f} рад/с")
axes[0].axhline(0, color=VIZ["edge"], linewidth=0.8)
axes[0].set_title("Закон движения гармонического осциллятора")
axes[0].set_xlabel("Время t (с)")
axes[0].set_ylabel("Перемещение x (м)")
axes[0].legend(loc="upper right")

# Фазовый портрет x(t) — v(t)
axes[1].plot(x_arr, v_arr, color=VIZ["series"][1])
axes[1].set_title("Фазовый портрет (x, v)")
axes[1].set_xlabel("Перемещение x (м)")
axes[1].set_ylabel("Скорость v (м/с)")

fig.tight_layout()
plt.show()

print(f"Символьное решение: x(t) = A·cos(sqrt(k/m)·t)")
print(f"Числовые параметры: A={A_val}, m={m_val}, k={k_val}, ω={omega_val:.3f} рад/с")

**Как читать эти графики.**

Левый — закон движения: перемещение как функция времени. Идеальный гармонический осциллятор колеблется бесконечно с постоянной амплитудой A и периодом $T = 2\pi / \omega$. Измените `k_val` — изменится частота колебаний; измените `A_val` — амплитуда.

Правый — фазовый портрет: совместная динамика перемещения и скорости. Для идеального осциллятора это эллипс (при одинаковых масштабах осей — окружность). Замкнутость траектории означает отсутствие диссипации: энергия не теряется. Для систем с затуханием эллипс превращается в закручивающуюся спираль к нулю.

## 9. Попробуйте на своих задачах

Несколько типичных применений в исследованиях:

- **Проверка аналитического результата**: вставьте свою формулу и вычислите производную или интеграл — сравните с вашим вычислением вручную.
- **Ряд Тейлора для вашей функции**: замените `sin(x)` на любую функцию и укажите нужный порядок.
- **Другая физическая система**: замените уравнение движения (добавьте затухание `2*gamma*x_fn(t).diff(t)` или внешнюю силу).
- **Экспорт формулы в код**: `lambdify` + `latex()` позволяют вставить символьно выведенную формулу прямо в статью или числовой решатель.

Ячейка ниже — заготовка для вашего ОДУ.

In [ ]:
# --- Шаблон: своё ОДУ первого порядка ---
# from sympy import Function, dsolve, Eq, diff, symbols, exp
#
# y_fn = Function("y")
# t_s  = symbols("t", positive=True)
# lam  = symbols("lambda", positive=True)
#
# # Ваше уравнение, например: dy/dt = -lambda*y (радиоактивный распад)
# my_ode = Eq(diff(y_fn(t_s), t_s) + lam * y_fn(t_s), 0)
# sol = dsolve(my_ode, y_fn(t_s))
# display(sol)
print("Шаблон готов. Раскомментируйте и адаптируйте под своё уравнение.")

## Что дальше

- Официальная документация SymPy: https://docs.sympy.org/
- Учебник SymPy (Tutorial): https://docs.sympy.org/latest/tutorial/index.html
- Для численного решения ОДУ после символьного вывода: `scipy.integrate.solve_ivp`.
- Для решения систем ОДУ с параметрами: `sympy.solvers.ode.systems`.
- Экспорт в C/Fortran-код: `sympy.codegen` и `lambdify(modules="numexpr")`.
- Связка SymPy + LaTeX: функция `latex(expr)` даёт готовый код для вставки в статью.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Вы ввели в SymPy выражение `(x**2 - 1) / (x - 1)` и применили `simplify`, получив `x + 1`. Можно ли утверждать, что оба выражения тождественно равны на всей числовой прямой, и какую проверку стоит сделать прежде чем подставлять результат в физическую модель?
2. SymPy вычисляет определённый интеграл `integrate(f, (x, 0, oo))` и возвращает `oo` (бесконечность). Числовая квадратура в scipy даёт конечный большой результат. Что может быть причиной расхождения и как это проверить?
3. Вы использовали `lambdify` для получения числовой функции из символьного решения ОДУ, но при подстановке массива `t_arr` функция возвращает массив нулей вместо ожидаемых значений. Назовите наиболее вероятную причину и способ её устранения.

<details>
<summary>Показать ориентиры для ответов</summary>

1. Выражения не тождественны: исходное не определено при `x = 1`, тогда как `x + 1` определено всюду. `simplify` проводит алгебраическое упрощение, не отслеживая области допустимых значений. Перед подстановкой в модель нужно явно задать ограничения через `assumptions` или отдельно проверить граничные точки.
2. Наиболее частые причины: интеграл расходится на бесконечности, но SymPy обнаруживает это, а scipy численно интегрирует до конечного предела; либо подынтегральная функция содержит ветвь, которую SymPy упростил иначе, чем scipy. Для проверки следует вычислить предел функции при `x → oo`, а также убедиться, что SymPy и scipy используют одно и то же выражение без неявных допущений о знаке переменных.
3. Наиболее вероятная причина: несовпадение аргументов `lambdify`. Если символьное решение содержит параметры (например, `m`, `k`, `A`), а в `lambdify` они не перечислены, функция возвращает константу. Нужно явно передать все свободные символы в первый аргумент `lambdify` в том же порядке, в котором они подставляются при вызове.
</details>
